## Alejandro Rodríguez Veloquio 746058
### Lab 03

In [1]:
from spark_utils import SparkUtils

su = SparkUtils("spark://spark-master:7077", "ARV-Lab 03")
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 01:58:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "int"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "float"),
    ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/quick_commerce_data_raw.csv")

qcommerce_df.show(5)

[Stage 0:>                                                          (0 + 1) / 1]

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3.2|
| 1000003|Flipkart Minute

In [3]:
from pyspark.sql.functions import when, count, isnull

qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [4]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(i), i)).alias(i) for i in qcommerce_df.columns])
qcommerce_null_count_df.show()

[Stage 1:=============================>                             (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [5]:
qcommerce_count2 = SparkUtils.count_nulls(qcommerce_df)

[Stage 4:=============================>                             (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
qcommerce_wo_nulls = qcommerce_df.dropna()
print(f"size of data frame after nemoving nulls: {qcommerce_wo_nulls.count()}")

[Stage 7:=============================>                             (1 + 1) / 2]

size of data frame after nemoving nulls: 780992


In [7]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    'City': 'Unknown',
    'Items_count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})

print(f"Size of data frame afer removing nulls with 'fillna': {qcommerce_wo_nulls_fillna.count()}")


Size of data frame afer removing nulls with 'fillna': 1000000


In [8]:
from pyspark.sql.functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country",lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [9]:
from pyspark.sql.functions import col

qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_tax", col("Order_value")* 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

## Task 1

In [10]:
qcommerce_df_task1 = qcommerce_tax_df.withColumn(
    "Delivery_SLA",
    when(col("Delivery_Time_Min") <= 15, "Fast")
    .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON-TIME")
    .when(col("Delivery_Time_Min") > 20, "LATE")) \
    .filter(col("Delivery_SLA") == "LATE") \
    .orderBy(col("Delivery_Time_Min"), ascending=False)

qcommerce_df_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

[Stage 15:=============================>                            (1 + 1) / 2]

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

## Task 2

In [11]:
from pyspark.sql.functions import count, avg, col, when

df_task2 = qcommerce_df_task1.withColumn(
    "Effective_Order_Value",
    col("Order_Value") * (1 - col("Discount_Applied"))
)

df_task2 = df_task2.withColumn(
    "Price_Bucket",
    when(col("Effective_Order_Value") < 200, "LOW")
    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM")
    .when(col("Effective_Order_Value") > 500, "HIGH")
)

result_task2 = df_task2.groupBy("Price_Bucket").agg(
    count("*").alias("total_orders"),
    avg("Effective_Order_Value").alias("avg_effective_order_value")
).orderBy(col("avg_effective_order_value").desc())

result_task2.show()

[Stage 17:=============================>                            (1 + 1) / 2]

+------------+------------+-------------------------+
|Price_Bucket|total_orders|avg_effective_order_value|
+------------+------------+-------------------------+
|        HIGH|       56013|         707.219837039914|
|      MEDIUM|       59824|       353.49134017350553|
|         LOW|      143811|        25.36497016341368|
+------------+------------+-------------------------+



## Task 3

In [12]:
from pyspark.sql.functions import col, when, count, avg

df_task3 = df_task2.filter(
    (col("Customer_Age").isNotNull()) &
    (col("Customer_Age") >= 0) &
    (col("Customer_Age") <= 100)
)

df_task3 = df_task3.withColumn(
    "Age_Group",
    when(col("Customer_Age") < 25, "YOUNG")
    .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT")
    .otherwise("SENIOR")
)

result_task3 = df_task3.groupBy("Age_Group", "Product_Category").agg(
    count("*").alias("orders"),
    avg("Order_Value").alias("avg_order_value")
)

result_task3 = result_task3.orderBy(
    col("Age_Group").asc(),
    col("orders").desc()
)

result_task3.show()

[Stage 24:=============================>                            (1 + 1) / 2]

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|orders|   avg_order_value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17935|495.73422188930334|
|    ADULT|              Dairy| 17780| 502.4952578519943|
|    ADULT|          Household| 17767|497.23453261697006|
|    ADULT|             Snacks| 17700|499.79997128481244|
|    ADULT|Fruits & Vegetables| 17671|494.20056196792063|
|    ADULT|          Beverages| 17650| 496.7228836926252|
|    ADULT|          Groceries| 17545|491.13630949723205|
|   SENIOR|          Groceries| 13440|500.87560482649576|
|   SENIOR|      Personal Care| 13253| 501.4177858687559|
|   SENIOR|             Snacks| 13250|502.81940932795686|
|   SENIOR|              Dairy| 13176|498.58672649583895|
|   SENIOR|          Household| 13163|  489.620170382284|
|   SENIOR|          Beverages| 13118|492.35374550340305|
|   SENIOR|Fruits & Vegetables| 12955|498.62873171412866|
|    YOUNG|   

In [13]:
su._spark.stop()